# NOAA before/after test pairs

Run after `python -m reefwatch.data.ingest_noaa`. Test data only: never train, tune or calibrate on it.

- **Site pairs**: cover change per functional group between consecutive visits (~300 points each). Reliable.
- **Photo pairs**: same site, replicate and photo number. Approximate, because the tape is re-laid from a permanent stake each visit.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from omegaconf import OmegaConf

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
cfg = OmegaConf.load(ROOT / "configs/data.yaml")
interim = ROOT / cfg.paths.interim
points = pd.read_parquet(interim / "noaa_points.parquet")
site_pairs = pd.read_parquet(interim / "noaa_site_pairs.parquet")
photo_pairs = pd.read_parquet(interim / "noaa_photo_pairs.parquet")
print(f"points {len(points):,} | sites {points.SITE.nunique():,} | photos {points.Image_name.nunique():,}")
print("years:", sorted(points.survey_year.dropna().unique().tolist()))

## Labels
Check that every common `raw_label` has an `l1` group. Anything with empty `l1` needs a line in `configs/taxonomy.yaml`.

In [ ]:
points.groupby(["raw_label", "l1"], dropna=False).size().sort_values(ascending=False).head(30)

## Point positions
The dictionary doesn't give units for `x_pos`/`y_pos`. If the maximum looks like pixel sizes (thousands), they are pixels; if it is at most 1, they are fractions of the image.

In [ ]:
points[["x_pos", "y_pos"]].describe()

## Visits per site

In [ ]:
points.groupby("SITE").survey_year.nunique().value_counts().sort_index()

## Site pairs: hard-coral cover change

In [ ]:
print(len(site_pairs), "site pairs")
if "delta_hard_coral" in site_pairs:
    site_pairs["delta_hard_coral"].mul(100).plot.hist(bins=40, title="Change in hard-coral cover (percentage points)")
    plt.show()
site_pairs.head()

## Photo pairs

In [ ]:
print(len(photo_pairs), "photo pairs")
photo_pairs.head()